In [23]:
import cv2 as cv
import numpy as np
import pandas as pd
import os
import time
from video_to_frames import video_to_frames

**Parameters**

In [24]:
# Frame params
FRAME_HEIGHT = 480
FRAME_WIDTH = 640

num_bins_x = 3
num_bins_y = 1

# Keypoint params
max_kpoints = 500
response_cutoff = 0.000

# Number of frames to process
n_frames = 50

In [25]:
# Create output folder
output_folder = "orb_keypoints"
os.makedirs(output_folder, exist_ok=True)

# Initiate ORB detector
orb = cv.ORB_create(nfeatures=max_kpoints)

# Storing keypoint data
kp_list = []

# Process images
for i in range(1, n_frames + 1):
    img_path = rf"Images\img{i:04d}.png"
    
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Failed to load {img_path}")
        continue
    
    # Detect and compute keypoints
    kp = orb.detect(img, None)
    kp, des = orb.compute(img, kp)

    # Filter keypoints by response
    kp_filtered = [k for k in kp if k.response >= response_cutoff]

    # Save raw keypoints
    kp_list.append(kp_filtered)
    
    # Draw keypoints
    img_kp = cv.drawKeypoints(img, kp_filtered, None, color=(0,255,0), flags=0)
    
    # Draw grid
    bin_width = FRAME_WIDTH // num_bins_x
    bin_height = FRAME_HEIGHT // num_bins_y
    
    # Vertical lines
    for x in range(0, FRAME_WIDTH + 1, bin_width):
        cv.line(img_kp, (x, 0), (x, FRAME_HEIGHT), (0, 0, 255), 2)
    
    # Horizontal lines
    for y in range(0, FRAME_HEIGHT + 1, bin_height):
        cv.line(img_kp, (0, y), (FRAME_WIDTH, y), (0, 0, 255), 2)
    
    # Save result
    output_path = os.path.join(output_folder, f'kp_img{i:04d}.png')
    cv.imwrite(output_path, img_kp)

print(f"\nAll images saved to: {output_folder}")


All images saved to: orb_keypoints


**Bin point count**

In [26]:
x_data = []
y_data = []
for frame in kp_list:
    x_data.extend([kp.pt[0] for kp in frame if kp.response >= response_cutoff])
    y_data.extend([kp.pt[1] for kp in frame if kp.response >= response_cutoff])

hist, x_edges, y_edges = np.histogram2d(x=x_data, y=y_data, bins=[num_bins_x, num_bins_y], range=[[0, FRAME_HEIGHT], [0, FRAME_WIDTH]])

print("Total point count per cell:")

for i in range(len(x_edges) - 1):
    for j in range(len(y_edges) - 1):
        print(f"Point count [[{x_edges[i]:.2f}, {x_edges[i+1]:.2f}], \t [{y_edges[j]:.2f}, {y_edges[j+1]:.2f}]]: \t{hist[i, j]}")


Total point count per cell:
Point count [[0.00, 160.00], 	 [0.00, 640.00]]: 	4801.0
Point count [[160.00, 320.00], 	 [0.00, 640.00]]: 	5768.0
Point count [[320.00, 480.00], 	 [0.00, 640.00]]: 	5197.0


In [27]:
from frame_processing_and_encoding import FrameProcessor

####### Testing ########
start = time.perf_counter()
spike_trains = []

# Initialize processor
thresholds = (0, 5, 15, 30, 100, 500)
processor = FrameProcessor(threshold_edges=thresholds, n_bins_x=num_bins_x, n_bins_y=num_bins_y)

# Loop through n_frame images
for idx in range(1, n_frames+1):
    img_path = rf"Images\img{idx:04d}.png"
    
    # Returns spike train per frame
    spike_train = processor.process_and_encode_frame(img_path)
    spike_trains.append(spike_train)
    
    #print(f"Frame {idx:04d} results:")
    #print(f"Spike Train: {spike_train}")
    #print("-" * 30)

end = time.perf_counter()
avg_time_per_frame = (end - start)/n_frames
print(f'Processing time per frame: {avg_time_per_frame:.6f} seconds')

df_results = pd.DataFrame(spike_trains)
df_results.to_csv("ORB_test_spiketrains.csv", index=False)

Processing time per frame: 0.004501 seconds
